# EnterpriseEngine tour

This offline tour loads component declarations, validates a small factory manifest, and runs a tiny simulation. It does not start services or make network calls.

In [1]:
from components.catalog import manifest, registry


def instance(instance_id, type_id, config, state, relations=None):
    return {
        'id': instance_id, 'type': type_id, 'config': config, 'state': state,
        'parameters': {}, 'policies': {}, 'relations': relations or {},
    }


instances = [
    instance('operator', 'norvale.role.v1', {'name': 'Operator'}, {'active': True}),
    instance('warehouse', 'factory.warehouse.v1', {'name': 'Finished goods'}, {'on_hand': 0}),
    instance('station', 'factory.workstation.v1', {'name': 'Mixer', 'capacity': 1}, {'available': True}),
    instance('work', 'factory.work_order.v1', {'number': 'WO-1', 'quantity': 4}, {'completed': 0}, {'workstation': 'station'}),
    instance('quality', 'factory.quality_check.v1', {'specification': 'pH'}, {'passed': False}, {'work_order': 'work'}),
]
resolved = registry().resolve(manifest(instances))
[(name, row['descriptor']['type']) for name, row in resolved.items()]

[('operator', 'norvale.role.v1'),
 ('warehouse', 'factory.warehouse.v1'),
 ('station', 'factory.workstation.v1'),
 ('work', 'factory.work_order.v1'),
 ('quality', 'factory.quality_check.v1')]

The registry confirms that every ID, saved-state value and relation matches its declaration before the engine starts.

In [2]:
from tempfile import TemporaryDirectory
from time import monotonic

from engine import Engine


class FactoryDesk:
    config = {}

    def handle(self, run, kind, payload):
        if kind == 'make':
            run.state['completed'] = payload['quantity']
        elif kind == 'inspect':
            run.state['quality'] = 'passed'
        else:
            raise ValueError(kind)


with TemporaryDirectory() as run_dir:
    run = Engine(run_dir, FactoryDesk(), seed=7, native=None, providers=None, monotonic=monotonic)
    run.extend(
        'tour-v1',
        roles={'operator': {'shifts': [[0, 60]], 'staff': 1}},
        events=[
            {'id': 'make', 'kind': 'make', 'at': 0, 'duration': 10, 'role': 'operator', 'payload': {'quantity': 4}},
            {'id': 'inspect', 'kind': 'inspect', 'at': 10, 'duration': 5, 'role': 'operator', 'payload': {}},
        ],
    )
    run.run(until=20)
    result = {'state': run.state, 'history': run.history(), 'records': run.export()['records']}
result

{'state': {'completed': 4, 'quality': 'passed'},
 'history': [{'at': 0, 'id': 'make', 'phase': 'start'},
  {'at': 10, 'id': 'make', 'phase': 'complete'},
  {'at': 10, 'id': 'inspect', 'phase': 'start'},
  {'at': 15, 'id': 'inspect', 'phase': 'complete'}],
 'records': []}

The engine owns the event order and checkpoint; the consumer supplies the small business actions. Components stay reusable because their declarations are separate from this Norvale-flavoured example.